# Phase 3 Part 3: SimGCL on All 5 Datasets
**CSC14114 - Big Data Applications | QRec-based SimGCL workflow**

This notebook completes **Phase 3** by running **SimGCL** on all 5 datasets with the QRec implementation, then comparing the three models from Part 1, Part 2, and Part 3.

## Phase 3 split
1. Part 1: run **LightGCN** on all 5 datasets
2. Part 2: run **SGL** on all 5 datasets
3. Part 3: run **SimGCL** on all 5 datasets and compare all 3 models

## What this notebook does
- prepares the same 5 datasets in the QRec format
- runs **3 repeated runs with fixed seed 42** per dataset
- uses SimGCL from the QRec fork repo
- reports `Recall@20`, `NDCG@20`, `HR@20`, and `MRR@20`
- loads the Part 1 and Part 2 summaries and builds a combined comparison table
- saves per-run outputs, averaged results, plots, and an archive

## Datasets used in Part 3
- `ml-1m`
- `gowalla`
- `yelp2018`
- `amazon-book`
- `lastfm_phase2`

## Notes
- The notebook clones `https://github.com/nguyenvmthien/QRec.git` at runtime.
- Part 1 and Part 2 should be run first if you want full 3-model comparison tables.


In [ ]:
import json
import os
import random
import shutil
import subprocess
import sys
import time
import zipfile
from contextlib import contextmanager
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import tensorflow.compat.v1 as tf

tf.disable_v2_behavior()

CODE_DIR = Path.cwd()
PART3_ROOT = CODE_DIR / 'phase3_part3_simgcl'
WORK_DIR = PART3_ROOT / 'workspace'
RESULTS_DIR = PART3_ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
AVG_RESULTS_DIR = RESULTS_DIR / 'average_results_3_runs'
TEMP_CONF_DIR = PART3_ROOT / 'configs'
QREC_REPO_URL = 'https://github.com/nguyenvmthien/QRec.git'
QREC_DIR = PART3_ROOT / 'QRec'

PART1_RESULTS_DIR = CODE_DIR / 'phase3_part1_lightgcn' / 'results'
PART2_RESULTS_DIR = CODE_DIR / 'phase3_part2_sgl' / 'results'

for directory in [WORK_DIR, RESULTS_DIR, FIG_DIR, AVG_RESULTS_DIR, TEMP_CONF_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def ensure_qrec_repo() -> Path:
    if QREC_DIR.exists() and (QREC_DIR / 'main.py').exists():
        return QREC_DIR
    if QREC_DIR.exists():
        shutil.rmtree(QREC_DIR)
    print(f'Cloning QRec into {QREC_DIR} ...')
    subprocess.run(['git', 'clone', '--depth', '1', QREC_REPO_URL, str(QREC_DIR)], check=True)
    return QREC_DIR

QREC_DIR = ensure_qrec_repo()
if str(QREC_DIR) not in sys.path:
    sys.path.insert(0, str(QREC_DIR))

from QRec import QRec
from util.config import ModelConf
from model.ranking.SimGCL import SimGCL

FIXED_SEED = 42
RUN_IDS = [1, 2, 3]
DATASETS = ['ml-1m', 'gowalla', 'yelp2018', 'amazon-book', 'lastfm_phase2']
REPORT_TOPK = 20
TOPKS = [20]

print('Notebook dir:', CODE_DIR)
print('QRec dir:', QREC_DIR)
print('Part 1 results dir:', PART1_RESULTS_DIR)
print('Part 2 results dir:', PART2_RESULTS_DIR)
print('Part 3 results dir:', RESULTS_DIR)
print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.test.is_gpu_available(cuda_only=False, min_cuda_compute_capability=None)))

def set_global_seed(seed: int = FIXED_SEED) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.set_random_seed(seed)

set_global_seed(FIXED_SEED)

In [ ]:
LASTFM_REQUIRED_FILES = ['interactions.csv', 'users.csv', 'items.csv', 'manifest.json']
MOVIELENS_URL = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'
RAW_SPLIT_URLS = {
    'gowalla': {
        'train': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/gowalla/train.txt',
        'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/gowalla/test.txt',
    },
    'yelp2018': {
        'train': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/yelp2018/train.txt',
        'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/yelp2018/test.txt',
    },
    'amazon-book': {
        'train': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/amazon-book/train.txt',
        'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/amazon-book/test.txt',
    },
}

def download_file(url: str, dest: Path) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f'[skip] {dest.name}')
        return dest
    print(f'[download] {url}')
    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()
    with dest.open('wb') as handle:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                handle.write(chunk)
    return dest

def parse_lightgcn_split(path: Path):
    pairs = []
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            parts = line.strip().split()
            if not parts:
                continue
            user_id = parts[0]
            for item_id in parts[1:]:
                pairs.append((user_id, item_id))
    return pairs

def write_qrec_split(path: Path, interactions):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        for user_id, item_id in interactions:
            handle.write(f'{user_id} {item_id} 1\n')

def build_leave_one_out_maps(df: pd.DataFrame, user_col: str, item_col: str, time_col: str):
    work = df[[user_col, item_col, time_col]].copy()
    work = work.sort_values([user_col, time_col, item_col], kind='mergesort')
    users = sorted(work[user_col].unique().tolist())
    items = sorted(work[item_col].unique().tolist())
    user_map = {user: idx for idx, user in enumerate(users)}
    item_map = {item: idx for idx, item in enumerate(items)}
    work['uid'] = work[user_col].map(user_map).astype(int)
    work['iid'] = work[item_col].map(item_map).astype(int)

    train_map = {}
    test_map = {}
    for uid, group in work.groupby('uid', sort=True):
        item_list = group['iid'].tolist()
        if len(item_list) < 2:
            continue
        train_map[int(uid)] = [int(item) for item in item_list[:-1]]
        test_map[int(uid)] = [int(item_list[-1])]
    return train_map, test_map

def maps_to_pairs(data_map: dict):
    pairs = []
    for user_id in sorted(data_map):
        for item_id in data_map[user_id]:
            pairs.append((str(user_id), str(item_id)))
    return pairs

def find_lastfm_phase2_source():
    search_roots = [Path('/kaggle/input'), PART3_ROOT, CODE_DIR]
    for root in search_roots:
        if not root.exists():
            continue
        for zip_path in sorted(root.rglob('*.zip')):
            try:
                with zipfile.ZipFile(zip_path) as archive:
                    names = {Path(name).name for name in archive.namelist() if not name.endswith('/')}
                if all(name in names for name in LASTFM_REQUIRED_FILES):
                    return ('zip', zip_path)
            except zipfile.BadZipFile:
                continue
        for manifest_path in sorted(root.rglob('manifest.json')):
            folder = manifest_path.parent
            if all((folder / name).exists() for name in LASTFM_REQUIRED_FILES):
                return ('dir', folder)
    raise FileNotFoundError(
        'Could not find lastfm_phase2 input data. Attach a Kaggle dataset or place the processed files in the workspace.'
    )

def materialize_lastfm_phase2_input() -> Path:
    target_dir = WORK_DIR / 'lastfm_phase2_input'
    if target_dir.exists() and all((target_dir / name).exists() for name in LASTFM_REQUIRED_FILES):
        return target_dir

    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    source_kind, source_path = find_lastfm_phase2_source()
    if source_kind == 'zip':
        with zipfile.ZipFile(source_path) as archive:
            archive.extractall(target_dir)
    else:
        for name in LASTFM_REQUIRED_FILES:
            shutil.copy2(source_path / name, target_dir / name)
    return target_dir

LASTFM_PHASE2_INPUT_DIR = materialize_lastfm_phase2_input()

def prepare_qrec_datasets():
    for dataset_name, urls in RAW_SPLIT_URLS.items():
        dataset_dir = QREC_DIR / 'dataset' / dataset_name
        train_file = dataset_dir / 'train.txt'
        test_file = dataset_dir / 'test.txt'
        if train_file.exists() and test_file.exists():
            print(f'{dataset_name} already prepared')
            continue
        train_path = download_file(urls['train'], WORK_DIR / 'downloads' / dataset_name / 'train.txt')
        test_path = download_file(urls['test'], WORK_DIR / 'downloads' / dataset_name / 'test.txt')
        train_pairs = parse_lightgcn_split(train_path)
        test_pairs = parse_lightgcn_split(test_path)
        write_qrec_split(train_file, train_pairs)
        write_qrec_split(test_file, test_pairs)
        print(f'Prepared {dataset_name}')

    ml_1m_dir = QREC_DIR / 'dataset' / 'ml-1m'
    ml_1m_train = ml_1m_dir / 'train.txt'
    ml_1m_test = ml_1m_dir / 'test.txt'
    if not (ml_1m_train.exists() and ml_1m_test.exists()):
        zip_path = download_file(MOVIELENS_URL, WORK_DIR / 'downloads' / 'ml-1m.zip')
        extract_dir = WORK_DIR / 'downloads' / 'ml-1m_extracted'
        if not extract_dir.exists():
            with zipfile.ZipFile(zip_path) as archive:
                archive.extractall(extract_dir)
        ratings_path = extract_dir / 'ml-1m' / 'ratings.dat'
        df = pd.read_csv(
            ratings_path,
            sep='::',
            header=None,
            names=['user_id', 'item_id', 'rating', 'timestamp'],
            engine='python',
            encoding='latin-1',
        )
        df = df[['user_id', 'item_id', 'timestamp']].drop_duplicates(['user_id', 'item_id'])
        train_map, test_map = build_leave_one_out_maps(df, 'user_id', 'item_id', 'timestamp')
        write_qrec_split(ml_1m_train, maps_to_pairs(train_map))
        write_qrec_split(ml_1m_test, maps_to_pairs(test_map))
        print('Prepared ml-1m')

    lastfm_dir = QREC_DIR / 'dataset' / 'lastfm_phase2'
    lastfm_train = lastfm_dir / 'train.txt'
    lastfm_test = lastfm_dir / 'test.txt'
    if not (lastfm_train.exists() and lastfm_test.exists()):
        df = pd.read_csv(LASTFM_PHASE2_INPUT_DIR / 'interactions.csv')
        df['timestamp'] = pd.to_datetime(df['crawl_timestamp_utc'], utc=True, errors='coerce')
        df = df.dropna(subset=['timestamp']).copy()
        df['timestamp'] = df['timestamp'].astype('int64') // 10**9
        df = df[['user_id', 'item_id', 'timestamp']].drop_duplicates(['user_id', 'item_id'])
        train_map, test_map = build_leave_one_out_maps(df, 'user_id', 'item_id', 'timestamp')
        write_qrec_split(lastfm_train, maps_to_pairs(train_map))
        write_qrec_split(lastfm_test, maps_to_pairs(test_map))
        print('Prepared lastfm_phase2')

prepare_qrec_datasets()

print('Last.fm Phase 2 input:', LASTFM_PHASE2_INPUT_DIR)
print('Prepared QRec datasets under:', QREC_DIR / 'dataset')


In [ ]:
@contextmanager
def pushd(path: Path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)

def build_simgcl_conf(dataset_name: str, run_dir: Path) -> Path:
    dataset_dir = QREC_DIR / 'dataset' / dataset_name
    output_dir = run_dir / 'outputs'
    output_dir.mkdir(parents=True, exist_ok=True)
    conf_path = TEMP_CONF_DIR / f'SimGCL_{dataset_name}_{run_dir.name}.conf'
    conf_text = '\n'.join([
        f'ratings={dataset_dir / "train.txt"}',
        'ratings.setup=-columns 0 1 2',
        'model.name=SimGCL',
        f'evaluation.setup=-testSet {dataset_dir / "test.txt"} -b 1',
        'item.ranking=on -topN 20',
        'num.factors=64',
        'num.max.epoch=200',
        'batch_size=2048',
        'learnRate=-init 0.001 -max 1',
        'reg.lambda=-u 0.0001 -i 0.0001 -b 0.2 -s 0.2',
        'SimGCL=-n_layer 2 -lambda 0.5 -eps 0.1',
        f'output.setup=on -dir {output_dir.as_posix()}/',
    ])
    conf_path.write_text(conf_text, encoding='utf-8')
    return conf_path

def ranking_metrics_from_model(model_obj, topk: int = REPORT_TOPK):
    def user_metrics(ranked_items, truth_items, k):
        hits = [1.0 if item in truth_items else 0.0 for item in ranked_items[:k]]
        precision = sum(hits) / k
        recall = sum(hits) / len(truth_items) if truth_items else 0.0
        hr = 1.0 if any(hits) else 0.0
        dcg = 0.0
        for idx, hit in enumerate(hits):
            if hit:
                dcg += 1.0 / np.log2(idx + 2.0)
        ideal_len = min(len(truth_items), k)
        idcg = sum(1.0 / np.log2(idx + 2.0) for idx in range(ideal_len))
        ndcg = dcg / idcg if idcg > 0 else 0.0
        mrr = 0.0
        for idx, hit in enumerate(hits):
            if hit:
                mrr = 1.0 / (idx + 1.0)
                break
        return precision, recall, ndcg, hr, mrr

    user_test_dict = model_obj.data.testSet_u
    users = list(user_test_dict.keys())
    totals = {
        'precision@20': 0.0,
        'recall@20': 0.0,
        'ndcg@20': 0.0,
        'hr@20': 0.0,
        'mrr@20': 0.0,
    }

    for user in users:
        scores = np.asarray(model_obj.predictForRanking(user), dtype=np.float64).copy()
        rated_list, _ = model_obj.data.userRated(user)
        for item_id in rated_list:
            scores[model_obj.data.item[item_id]] = -np.inf

        if len(scores) == 0:
            continue

        k = min(topk, len(scores))
        top_idx = np.argpartition(-scores, kth=k - 1)[:k]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        ranked_items = [model_obj.data.id2item[idx] for idx in top_idx]
        truth_items = set(user_test_dict[user].keys())
        precision, recall, ndcg, hr, mrr = user_metrics(ranked_items, truth_items, topk)
        totals['precision@20'] += precision
        totals['recall@20'] += recall
        totals['ndcg@20'] += ndcg
        totals['hr@20'] += hr
        totals['mrr@20'] += mrr

    user_count = max(len(users), 1)
    return {key: value / user_count for key, value in totals.items()}

def run_simgcl_once(dataset_name: str, run_id: int):
    run_name = f'run_{run_id}'
    run_dir = RESULTS_DIR / dataset_name / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    conf_path = build_simgcl_conf(dataset_name, run_dir)

    set_global_seed(FIXED_SEED)
    tf.reset_default_graph()
    started_at = time.time()

    with pushd(QREC_DIR):
        qrec_loader = QRec(ModelConf(str(conf_path)))
        model_obj = SimGCL(ModelConf(str(conf_path)), qrec_loader.trainingData, qrec_loader.testData)
        measure = model_obj.execute()

    elapsed_seconds = round(time.time() - started_at, 2)
    metrics = ranking_metrics_from_model(model_obj, REPORT_TOPK)

    output_dir = run_dir / 'outputs'
    measure_files = sorted(output_dir.glob('*measure*.txt'))
    if not measure_files:
        raise RuntimeError(f'No measure file produced for {dataset_name} {run_name}.')
    shutil.copy2(measure_files[-1], run_dir / 'measure.txt')

    log_dir = QREC_DIR / 'log'
    log_files = sorted(log_dir.glob('*.log'))
    if log_files:
        shutil.copy2(log_files[-1], run_dir / 'train.log')

    result = {
        'dataset': dataset_name,
        'model': 'SimGCL',
        'run_id': run_id,
        'run_name': run_name,
        'seed': FIXED_SEED,
        'elapsed_seconds': elapsed_seconds,
        'gpu': 'tensorflow-v1',
        **metrics,
    }
    (run_dir / 'metrics.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    (run_dir / 'measure_lines.json').write_text(json.dumps(measure, indent=2), encoding='utf-8')
    (run_dir / 'run_complete.json').write_text(
        json.dumps({
            'dataset': dataset_name,
            'run_id': run_id,
            'status': 'completed',
            'completed_at_unix': int(time.time()),
        }, indent=2),
        encoding='utf-8',
    )
    return result

print('SimGCL runner is ready.')

## Run all Phase 3 Part 3 experiments
This launches **SimGCL** on all 5 datasets with **3 repeated runs** and stores the per-run outputs under the Part 3 results folder.

In [ ]:
all_results = []
for dataset_name in DATASETS:
    for run_id in RUN_IDS:
        run_dir = RESULTS_DIR / dataset_name / f'run_{run_id}'
        metrics_path = run_dir / 'metrics.json'
        measure_path = run_dir / 'measure.txt'
        complete_path = run_dir / 'run_complete.json'

        is_completed_run = metrics_path.exists() and measure_path.exists() and complete_path.exists()
        has_partial_artifacts = run_dir.exists() and not is_completed_run and any(run_dir.iterdir())

        if is_completed_run:
            cached = json.loads(metrics_path.read_text(encoding='utf-8'))
            all_results.append(cached)
            print(f'[skip] {dataset_name} run_{run_id} already completed')
            continue

        if has_partial_artifacts:
            print(f'[clean] remove partial artifacts for {dataset_name} run_{run_id}')
            shutil.rmtree(run_dir)
            run_dir.mkdir(parents=True, exist_ok=True)

        print(f'[run ] {dataset_name} run_{run_id}')
        all_results.append(run_simgcl_once(dataset_name, run_id))

results_df = pd.DataFrame(all_results).sort_values(['dataset', 'run_id']).reset_index(drop=True)
results_df.to_csv(RESULTS_DIR / 'all_runs.csv', index=False)
display(results_df)


## Build the Part 3 summary and the cross-model comparison
This section creates the averaged SimGCL report and then merges the Part 1 and Part 2 summaries into a single three-model comparison table.

In [ ]:
metric_cols = [
    'precision@20',
    'recall@20',
    'ndcg@20',
    'hr@20',
    'mrr@20',
    'elapsed_seconds',
]

summary_rows = []
for dataset_name, group in results_df.groupby('dataset', sort=True):
    row = {'dataset': dataset_name, 'model': 'SimGCL', 'runs': int(len(group))}
    for column in metric_cols:
        row[f'{column}_mean'] = float(group[column].mean())
        row[f'{column}_std'] = float(group[column].std(ddof=0))
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('dataset').reset_index(drop=True)
summary_df.to_csv(RESULTS_DIR / 'summary.csv', index=False)
summary_df.to_csv(AVG_RESULTS_DIR / 'summary_with_std.csv', index=False)

if (summary_df['runs'] != len(RUN_IDS)).any():
    raise RuntimeError('Cannot build 3-run averages because at least one dataset has missing SimGCL runs.')

average_report_df = pd.DataFrame({
    'Dataset': summary_df['dataset'],
    'Model': summary_df['model'],
    f'Recall@{REPORT_TOPK}': summary_df[f'recall@20_mean'].round(6),
    f'NDCG@{REPORT_TOPK}': summary_df[f'ndcg@20_mean'].round(6),
    f'HR@{REPORT_TOPK}': summary_df[f'hr@20_mean'].round(6),
    f'MRR@{REPORT_TOPK}': summary_df[f'mrr@20_mean'].round(6),
    'ElapsedSeconds': summary_df['elapsed_seconds_mean'].round(6),
})
average_report_df.to_csv(AVG_RESULTS_DIR / 'average_metrics_3_runs.csv', index=False)
(AVG_RESULTS_DIR / 'average_metrics_3_runs.json').write_text(json.dumps(average_report_df.to_dict(orient='records'), indent=2), encoding='utf-8')

for row in summary_df.to_dict(orient='records'):
    dataset_payload = {
        'dataset': row['dataset'],
        'model': row['model'],
        'runs': row['runs'],
        'mean': {
            f'recall@{REPORT_TOPK}': row['recall@20_mean'],
            f'ndcg@{REPORT_TOPK}': row['ndcg@20_mean'],
            f'hr@{REPORT_TOPK}': row['hr@20_mean'],
            f'mrr@{REPORT_TOPK}': row['mrr@20_mean'],
            'elapsed_seconds': row['elapsed_seconds_mean'],
        },
        'std': {
            f'recall@{REPORT_TOPK}': row['recall@20_std'],
            f'ndcg@{REPORT_TOPK}': row['ndcg@20_std'],
            f'hr@{REPORT_TOPK}': row['hr@20_std'],
            f'mrr@{REPORT_TOPK}': row['mrr@20_std'],
            'elapsed_seconds': row['elapsed_seconds_std'],
        },
    }
    (AVG_RESULTS_DIR / f"{row['dataset']}_average.json").write_text(json.dumps(dataset_payload, indent=2), encoding='utf-8')

    dataset_avg_dir = RESULTS_DIR / row['dataset'] / 'average_results_3_runs'
    dataset_avg_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([
        {
            'Dataset': row['dataset'],
            'Model': row['model'],
            'Runs': row['runs'],
            'Recall@20_mean': round(row['recall@20_mean'], 6),
            'Recall@20_std': round(row['recall@20_std'], 6),
            'NDCG@20_mean': round(row['ndcg@20_mean'], 6),
            'NDCG@20_std': round(row['ndcg@20_std'], 6),
            'HR@20_mean': round(row['hr@20_mean'], 6),
            'HR@20_std': round(row['hr@20_std'], 6),
            'MRR@20_mean': round(row['mrr@20_mean'], 6),
            'MRR@20_std': round(row['mrr@20_std'], 6),
            'ElapsedSeconds_mean': round(row['elapsed_seconds_mean'], 6),
            'ElapsedSeconds_std': round(row['elapsed_seconds_std'], 6),
        }
    ]).to_csv(dataset_avg_dir / 'average_metrics_3_runs.csv', index=False)
    (dataset_avg_dir / 'average_metrics_3_runs.json').write_text(json.dumps(dataset_payload, indent=2), encoding='utf-8')

def load_model_summary(summary_dir: Path, model_name: str) -> pd.DataFrame:
    summary_path = summary_dir / 'summary.csv'
    if not summary_path.exists():
        raise FileNotFoundError(f'Missing summary file: {summary_path}. Run the matching notebook first.')
    df = pd.read_csv(summary_path).copy()
    if 'model' not in df.columns:
        df['model'] = model_name
    return df

lightgcn_summary = load_model_summary(PART1_RESULTS_DIR, 'LightGCN')
sgl_summary = load_model_summary(PART2_RESULTS_DIR, 'SGL')
simgcl_summary = summary_df.copy()

comparison_df = pd.concat([lightgcn_summary, sgl_summary, simgcl_summary], ignore_index=True, sort=False)
comparison_df = comparison_df.sort_values(['dataset', 'model']).reset_index(drop=True)
comparison_df.to_csv(RESULTS_DIR / 'comparison_summary.csv', index=False)

comparison_report_df = comparison_df[[
    'dataset', 'model', 'runs',
    'recall@20_mean', 'recall@20_std',
    'ndcg@20_mean', 'ndcg@20_std',
    'hr@20_mean', 'hr@20_std',
    'mrr@20_mean', 'mrr@20_std',
    'elapsed_seconds_mean', 'elapsed_seconds_std',
]].copy()
display(summary_df)
display(average_report_df)
display(comparison_report_df)


In [ ]:
comparison_report_df.to_csv(RESULTS_DIR / 'comparison_report.csv', index=False)
(RESULTS_DIR / 'comparison_report.json').write_text(
    json.dumps(comparison_report_df.to_dict(orient='records'), indent=2),
    encoding='utf-8',
 )
print('Saved comparison report files.')


In [ ]:
palette = {'LightGCN': '#2f4b7c', 'SGL': '#a05195', 'SimGCL': '#ff7c43'}
plot_metrics = [
    ('recall@20_mean', f'Recall@{REPORT_TOPK}'),
    ('ndcg@20_mean', f'NDCG@{REPORT_TOPK}'),
    ('hr@20_mean', f'HR@{REPORT_TOPK}'),
    ('mrr@20_mean', f'MRR@{REPORT_TOPK}'),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 10), constrained_layout=True)
for ax, (metric_name, title) in zip(axes.flat, plot_metrics):
    pivot = comparison_df.pivot(index='dataset', columns='model', values=metric_name).reindex(DATASETS)
    x = np.arange(len(pivot.index))
    bar_width = 0.24
    offsets = [-bar_width, 0.0, bar_width]
    for offset, model_name in zip(offsets, ['LightGCN', 'SGL', 'SimGCL']):
        if model_name not in pivot.columns:
            continue
        ax.bar(x + offset, pivot[model_name].values, width=bar_width, label=model_name, color=palette[model_name])
    ax.set_title(title, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=18)
    ax.grid(axis='y', alpha=0.2)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, frameon=False)
plt.savefig(FIG_DIR / 'model_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

run_metadata = {
    'fixed_seed': FIXED_SEED,
    'run_ids': RUN_IDS,
    'datasets': DATASETS,
    'report_topk': REPORT_TOPK,
    'qrec_dir': str(QREC_DIR),
    'lastfm_phase2_input': str(LASTFM_PHASE2_INPUT_DIR),
    'part1_results_dir': str(PART1_RESULTS_DIR),
    'part2_results_dir': str(PART2_RESULTS_DIR),
    'part3_results_dir': str(RESULTS_DIR),
}
(RESULTS_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')

archive_base = PART3_ROOT / 'phase3_part3_simgcl_results'
archive_zip = archive_base.with_suffix('.zip')
if archive_zip.exists():
    archive_zip.unlink()
shutil.make_archive(str(archive_base), 'zip', RESULTS_DIR)
print(f'Results archive: {archive_zip}')
print('Finished Phase 3 Part 3.')
